<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/firefoxPreprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openpyxl -q

import pandas as pd
import re
from google.colab import files

print("✅ Ready!")

✅ Ready!


In [2]:
print("📁 Upload Firefox_bugs.csv")
uploaded = files.upload()

df = pd.read_csv('Firefox_bugs.csv', low_memory=False)
print(f"✅ Loaded: {df.shape}")
print("\nOriginal Priority distribution:")
print(df['Priority'].value_counts(dropna=False))

📁 Upload Firefox_bugs.csv


Saving Firefox_bugs.csv to Firefox_bugs.csv
✅ Loaded: (24824, 8)

Original Priority distribution:
Priority
--    12174
P5     4654
P3     3791
P1     2325
P2     1823
P4       57
Name: count, dtype: int64


In [3]:
# Caddy & Treude (2024) PROMISE Paper Figure 1
# P1 → High, P2 → High, P3 → Medium, P4 → Low, P5 → Low
bugzilla_map = {
    'P1': 'High',
    'P2': 'High',
    'P3': 'Medium',
    'P4': 'Low',
    'P5': 'Low',
    '--': None
}

df['priority'] = df['Priority'].map(bugzilla_map)

# Drop unset rows (--)
df = df[df['priority'].notna()].copy()
df = df.reset_index(drop=True)

print(f"✅ After fixing Priority: {df.shape}")
print(df['priority'].value_counts())

✅ After fixing Priority: (12650, 9)
priority
Low       4711
High      4148
Medium    3791
Name: count, dtype: int64


In [4]:
# Combine Summary + Description
df['text'] = (
    df['Summary'].fillna('').str.strip() + ' ' +
    df['Description'].fillna('').str.strip()
)

print(f"✅ Text column created!")
print(f"Avg words (before clean): {df['text'].str.split().str.len().mean():.0f}")
print(f"Max words (before clean): {df['text'].str.split().str.len().max()}")

✅ Text column created!
Avg words (before clean): 321
Max words (before clean): 7325


In [5]:
def clean_text(text):
    text = str(text)

    # Remove log lines [task 2020-01-02T14:00:26Z] ...
    text = re.sub(
        r'\[task \d{4}-\d{2}-\d{2}T[\d:.]+Z\].*?(?=\[task|\Z)',
        '', text, flags=re.DOTALL
    )
    # Remove Bugzilla boilerplate
    text = re.sub(
        r'(User Agent|Build Identifier|Filed by|Parsed log|Full log)\s*:.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove test log lines
    text = re.sub(
        r'(TEST-PASS|TEST-OK|TEST-FAIL|TEST-UNEXPECTED|GECKO\(\d+\)|INFO\s*-)\s*.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove memory stat lines
    text = re.sub(r'MEMORY STAT.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    # Remove file paths
    text = re.sub(
        r'[\w/\-\.]+\.(js|cpp|py|java|html|css|ts|json|xml|md|log)\b',
        '', text, flags=re.IGNORECASE
    )
    # Remove code blocks
    text = re.sub(r'```.*?```', ' ', text, flags=re.DOTALL)
    text = re.sub(r'`[^`\n]+`', ' ', text)
    # Remove markdown bold/italic
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*\n]+)\*', r'\1', text)
    # Remove markdown links [text](url)
    text = re.sub(r'\[([^\]]+)\]\([^\)]*\)', r'\1', text)
    # Remove hex values
    text = re.sub(r'\b0x[0-9a-fA-F]+\b', '', text)
    # Remove timestamps
    text = re.sub(r'\b\d{2}:\d{2}:\d{2}(\.\d+)?\b', '', text)
    # Remove dates
    text = re.sub(r'\b\d{4}-\d{2}-\d{2}\b', '', text)
    # Remove version numbers
    text = re.sub(r'\bv?\d+\.\d+(\.\d+)*\b', '', text)
    # Remove separator lines
    text = re.sub(r'[-=*_#~^|<>]{3,}', ' ', text)

    # Remove noisy lines (no meaningful alpha content)
    lines = text.split('\n')
    clean_lines = []
    for line in lines:
        line = line.strip()
        if len(line) < 3:
            continue
        alpha_count = sum(c.isalpha() for c in line)
        if alpha_count < 3:
            continue
        if alpha_count / (len(line) + 1) < 0.3:
            continue
        clean_lines.append(line)

    text = ' '.join(clean_lines)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("⏳ Cleaning texts — please wait 1-2 mins...")
df['text'] = df['text'].apply(clean_text)
print("✅ Text cleaning done!")
print(f"Avg words (after clean): {df['text'].str.split().str.len().mean():.0f}")

⏳ Cleaning texts — please wait 1-2 mins...
✅ Text cleaning done!
Avg words (after clean): 73


In [6]:
# TinyBERT + CodeBERT max_length=128 tokens
# 100 words ≈ 110–120 tokens → safe range

def truncate(text, max_words=100):
    return ' '.join(text.split()[:max_words])

df['text'] = df['text'].apply(truncate)

print("✅ Truncated to max 100 words!")
print(f"Avg words (after truncate): {df['text'].str.split().str.len().mean():.0f}")
print(f"Max words (after truncate): {df['text'].str.split().str.len().max()}")

✅ Truncated to max 100 words!
Avg words (after truncate): 54
Max words (after truncate): 100


In [7]:
# Drop texts too short after cleaning
df = df[df['text'].str.split().str.len() >= 5].copy()

# Drop duplicates
df = df.drop_duplicates(subset='text').copy()
df = df.reset_index(drop=True)

print(f"✅ After quality filter: {df.shape}")
print(df['priority'].value_counts())

✅ After quality filter: (9811, 10)
priority
High      4090
Medium    3505
Low       2216
Name: count, dtype: int64


In [8]:
label_map = {'High': 0, 'Medium': 1, 'Low': 2}
df['priority_id'] = df['priority'].map(label_map)
df['source']      = 'gitbugs_firefox'

final = df[['text', 'priority', 'priority_id', 'source']].copy()

print("✅ Final columns set!")
print(final.head(3).to_string())

✅ Final columns set!
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     text priority  priority_id           source
0  Inefficient z-ordering of elements in the browser UI, causes large layers with current WebRender heuristics Steps to reproduce: 1. Be on macOS. 2. Enable the prefs , (requires restart), and (restartless). 3. Pay attention to the amount of redness on the screen. Expected results: The active tab and the back button should be drawn into the same red layer as the rest 

In [9]:
wc = final['text'].str.split().str.len()

print("=" * 50)
print("   ✅ FIREFOX_CLEAN — FINAL RESULTS")
print("=" * 50)
print(f"  Total rows     : {len(final):,}")
print(f"  Avg word count : {wc.mean():.0f}")
print(f"  Min word count : {wc.min()}")
print(f"  Max word count : {wc.max()}")
print()
print("  Priority distribution:")
print(final['priority'].value_counts().to_string())
print()
print("  Priority %:")
print((final['priority'].value_counts(normalize=True)*100).round(1).to_string())
print()
print("  Sample rows:")
for _, row in final.sample(3, random_state=42).iterrows():
    print(f"\n  [{row['priority']}] {row['text'][:120]}")
print()
print("=" * 50)

# Save & Download
final.to_csv('firefox_clean.csv', index=False)
files.download('firefox_clean.csv')
print("✅ Downloaded: firefox_clean.csv")

   ✅ FIREFOX_CLEAN — FINAL RESULTS
  Total rows     : 9,811
  Avg word count : 68
  Min word count : 5
  Max word count : 100

  Priority distribution:
priority
High      4090
Medium    3505
Low       2216

  Priority %:
priority
High      41.7
Medium    35.7
Low       22.6

  Sample rows:

  [High] [Windows 10] Addon context menu checkbox items show icons instead of checkboxes, are misaligned See attached screenshot.

  [Low] handoff copy paste does not work Steps to reproduce: copy url from iOS (anywhere) try to paste from macos paste in firef

  [Low] Intermittent | should_retry_after_failure - [should_retry_after_failure : 199] 0 == 1



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: firefox_clean.csv
